              Query (Q)           Retrieved Docs (D1, D2, ...)
                  ↓                        ↓
        ┌──────────────────┐      ┌────────────────────┐
        │   Query Encoder   │      │   Context Encoder   │
        └──────────────────┘      └────────────────────┘
                  ↓                        ↓
           Query Embeddings         Context Embeddings
                  ↘                        ↙
               ┌─────────────────────────────┐
               │       Modified Decoder      │
               │ (Cross-attn to both Q & D)  │
               └─────────────────────────────┘
                             ↓
                     Generated Answer


In [1]:
pip install transformers datasets faiss-cpu torch

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install -U sentence-transformers

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
import torch.nn as nn
from transformers import T5ForConditionalGeneration, T5EncoderModel, AutoTokenizer, EncoderDecoderCache
import faiss
import numpy as np

/root/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import torch
import torch.nn as nn
from transformers import T5ForConditionalGeneration, T5EncoderModel, AutoTokenizer
from transformers.modeling_outputs import BaseModelOutput
import faiss
import numpy as np

class DualEncoderRAG(nn.Module):
    def __init__(self, model_name='google/flan-t5-xl'):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        # Separate encoders for query and context
        self.query_encoder = T5EncoderModel.from_pretrained(model_name)
        self.context_encoder = T5EncoderModel.from_pretrained(model_name)

        # Full model including decoder
        self.decoder_model = T5ForConditionalGeneration.from_pretrained(model_name)

    def generate(self, query_input_ids, query_attention_mask,
                 context_input_ids, context_attention_mask,
                 max_length=50):

        query_outputs = self.query_encoder(
            input_ids=query_input_ids,
            attention_mask=query_attention_mask
        )

        context_outputs = self.context_encoder(
            input_ids=context_input_ids,
            attention_mask=context_attention_mask
        )

        encoder_hidden_states = torch.cat(
            [query_outputs.last_hidden_state, context_outputs.last_hidden_state], dim=1
        )
        combined_attention_mask = torch.cat(
            [query_attention_mask, context_attention_mask], dim=1
        )

        encoder_outputs = BaseModelOutput(last_hidden_state=encoder_hidden_states)

        generated_ids = self.decoder_model.generate(
            encoder_outputs=encoder_outputs,
            attention_mask=combined_attention_mask,
            max_length=max_length,
            num_beams=4,
            no_repeat_ngram_size=3,
            repetition_penalty=1.2,
            early_stopping=True
        )

        return generated_ids

class SimpleRetriever:
    def __init__(self, model_name='sentence-transformers/all-MiniLM-L6-v2'):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer(model_name)
        self.index = None
        self.passages = []

    def build_index(self, corpus):
        self.passages = corpus
        embeddings = self.model.encode(corpus, convert_to_tensor=False)
        self.index = faiss.IndexFlatL2(len(embeddings[0]))
        self.index.add(np.array(embeddings).astype('float32'))

    def retrieve(self, query, k=1):
        query_vec = self.model.encode([query])[0].astype('float32')
        D, I = self.index.search(np.array([query_vec]), k)
        return ["DOCUMENT: " + self.passages[i] for i in I[0]]

# Usage prototype
if __name__ == '__main__':
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = DualEncoderRAG().to(device)
    retriever = SimpleRetriever()

    # Corpus for retrieval
    corpus = [
        "France is a country in Europe.",
        "Paris is the capital and most populous city of France.",
        "Berlin is the capital of Germany."
    ]
    retriever.build_index(corpus)

    # Query
    query = "What is the capital of France?"
    retrieved_docs = retriever.retrieve(query, k=1)

    # Tokenization
    tokenizer = model.tokenizer
    query_inputs = tokenizer(query, return_tensors='pt', padding=True, truncation=True).to(device)
    context_inputs = tokenizer(retrieved_docs, return_tensors='pt', padding=True, truncation=True).to(device)

    # Generate output
    generated_ids = model.generate(
        query_input_ids=query_inputs['input_ids'],
        query_attention_mask=query_inputs['attention_mask'],
        context_input_ids=context_inputs['input_ids'],
        context_attention_mask=context_inputs['attention_mask'],
        max_length=50
    )

    output = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    print("Generated:", output)

Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  5.15it/s]


Generated: ['paris']


In [5]:
from sentence_transformers import SentenceTransformer

class SimpleRetriever:
    def __init__(self, model_name='sentence-transformers/all-MiniLM-L6-v2'):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer(model_name)
        self.index = None
        self.passages = []

    def build_index(self, corpus):
        self.passages = corpus
        embeddings = self.model.encode(corpus, convert_to_tensor=False)
        self.index = faiss.IndexFlatL2(len(embeddings[0]))
        self.index.add(np.array(embeddings).astype('float32'))

    def retrieve(self, query, k=1):
        query_vec = self.model.encode([query])[0].astype('float32')
        D, I = self.index.search(np.array([query_vec]), k)
        return [self.passages[i] for i in I[0]]

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DualEncoderRAG().to(device)
retriever = SimpleRetriever()

Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  5.21it/s]


In [7]:
# Corpus for retrieval
corpus = [
    "France is a country in Europe.",
    "Paris is the capital and most populous city of France.",
    "Berlin is the capital of Germany."
]
retriever.build_index(corpus)

In [8]:
# Query
query = "What is the capital of France?"
retrieved_docs = retriever.retrieve(query, k=1)

In [9]:
# Tokenization
tokenizer = model.tokenizer
query_inputs = tokenizer(query, return_tensors='pt', padding=True, truncation=True).to(device)
context_inputs = tokenizer(retrieved_docs, return_tensors='pt', padding=True, truncation=True).to(device)

In [ ]:

# Generate output
generated_ids = model.generate(
    encoder_outputs=encoder_outputs,
    attention_mask=combined_attention_mask,
    max_length=max_length,
    num_beams=4,
    no_repeat_ngram_size=3,
    repetition_penalty=1.2,
    early_stopping=True
)


In [138]:
output = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
print("Generated:", output)

Generated: ['Paris is the capital of France. the capital of France. What is the capital of France?']
